# Uni-MuMER - Kaggle 2xT4 + DagsHub

Train QLoRA, theo dõi metric trực tiếp bằng MLflow và lưu toàn bộ artifact lên DagsHub.

In [1]:
# 1. Cấu hình
import json
import os
import sys
import uuid
from pathlib import Path

from kaggle_secrets import UserSecretsClient

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"

BASE_YAML_CONFIG = "train/Uni-MuMER-train.yaml"

# Nên để YAML runtime trong /kaggle/working để không ghi đè YAML gốc
RUNTIME_YAML_CONFIG = "/kaggle/working/runtime_Uni-MuMER-train.yaml"

YAML_CONFIG = BASE_YAML_CONFIG
NOTEBOOK_PATH = "uni-mumer-kaggle-dagshub v4.ipynb"
OUTPUT_DIR = "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora"

DAGSHUB_USERNAME = "NhatPot"
DAGSHUB_REPO = "test-unimer"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
RUN_UUID = uuid.uuid4().hex

# Cho notebook import được module nội bộ trong repo, ví dụ scripts.runtime_yaml
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Cho các lệnh subprocess cũng thấy project
os.environ["PYTHONPATH"] = PROJECT_DIR

# Lấy DagsHub token từ Kaggle Secrets
DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
if not DAGSHUB_TOKEN:
    raise RuntimeError("Kaggle Secret DAGSHUB_TOKEN is missing")

os.environ.update({
    "PROJECT_DIR": PROJECT_DIR,
    "CONDA_DIR": CONDA_DIR,
    "ENV_DIR": ENV_DIR,
    "PYTHON": PYTHON,
    "BASE_YAML_CONFIG": BASE_YAML_CONFIG,
    "RUNTIME_YAML_CONFIG": RUNTIME_YAML_CONFIG,
    "YAML_CONFIG": YAML_CONFIG,
    "NOTEBOOK_PATH": NOTEBOOK_PATH,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_UUID": RUN_UUID,
    "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow",
    "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
    "MLFLOW_FLATTEN_PARAMS": "TRUE",
    "MLFLOW_TAGS": json.dumps({
        "run_uuid": RUN_UUID,
        "source": "kaggle",
        "task": "sft",
        "dataset": "parquet_crohme_train",
    }),
})

print(f"Run UUID: {RUN_UUID}")
print(f"MLflow: {os.environ['MLFLOW_TRACKING_URI']}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"PROJECT_DIR in sys.path: {PROJECT_DIR in sys.path}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")
print(f"runtime_yaml.py exists: {Path(PROJECT_DIR, 'scripts/runtime_yaml.py').exists()}")

Run UUID: 99f4ad6ddcbb4c53ae0f0fda13d2cfa2
MLflow: https://dagshub.com/NhatPot/test-unimer.mlflow
PROJECT_DIR: /kaggle/working/test-unimer
PROJECT_DIR in sys.path: True
PYTHONPATH: /kaggle/working/test-unimer
runtime_yaml.py exists: False


In [2]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$PYTHON" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -f -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version

PREFIX=/kaggle/working/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /kaggle/working/miniconda
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /kaggle/working/miniconda/envs/unimumer

  added / updated specs:
    - python=3.10


The following packages will



==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [3]:
%%bash
# 3. Lấy source code
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi

git -C "$PROJECT_DIR" rev-parse --short HEAD

69d68f9


Cloning into '/kaggle/working/test-unimer'...


In [12]:
%%bash
# 3.5 Download test data
set -e

cd /kaggle/working/test-unimer

pip install -q gdown

gdown "1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L" -O crohme_test_images.zip

unzip -q crohme_test_images.zip -d /kaggle/working/test-unimer

rm crohme_test_images.zip

ls -lah /kaggle/working/test-unimer/data

total 40K
drwxr-xr-x  8 root root 4.0K Aug 27  2025 .
drwxr-xr-x 13 root root 4.0K Jun 24 14:08 ..
drwxr-xr-x  7 root root 4.0K Jun  5  2025 CROHME
drwxr-xr-x  6 root root 4.0K Aug 26  2025 CROHME2023
-rw-r--r--  1 root root 6.1K Aug 27  2025 .DS_Store
drwxr-xr-x  5 root root 4.0K Aug 27  2025 HME100K
drwxr-xr-x  6 root root 4.0K Aug 27  2025 Im2LaTeXv2
drwxr-xr-x  5 root root 4.0K Aug 27  2025 MathWriting
drwxr-xr-x  7 root root 4.0K Jun  5  2025 MNE


Downloading...
From (original): https://drive.google.com/uc?id=1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L
From (redirected): https://drive.google.com/uc?id=1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L&confirm=t&uuid=b62cd15b-c393-4472-ba0e-fafa379542b2
To: /kaggle/working/test-unimer/crohme_test_images.zip
100%|██████████| 1.63G/1.63G [00:13<00:00, 123MB/s] 


In [5]:
%%bash
# 4. Cài dependency
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python --version
python -m pip install -q -r requirements.txt
python -m pip install -q -e train/LLaMA-Factory
python -c "import torch, mlflow; print('GPU:', torch.cuda.get_device_name(0)); print('MLflow:', mlflow.__version__)"

Python 3.10.20
GPU: Tesla T4
MLflow: 3.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
prometheus-fastapi-instrumentator 8.0.2 requires starlette<2.0.0,>=1.0.0, but you have starlette 0.52.1 which is incompatible.


In [6]:
# 5. Runtime YAML Override (tùy chọn)
from scripts.runtime_yaml import prepare_runtime_yaml

USE_RUNTIME_YAML_OVERRIDE = True

YAML_OVERRIDES = {
    "dataset": "parquet_crohme_train",
    "max_samples": 3,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 64,
    "learning_rate": 1.0e-4,
    "lora_rank": 64,
    "logging_steps": 1,
    "save_steps": 20,
    "output_dir": "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora",
    
    # Để None nếu không muốn override
    "cutoff_len": None,
    "val_size": None,
    "per_device_eval_batch_size": None,
    "eval_steps": None,
    "save_total_limit": None,
    "lora_alpha": None,
    "lora_dropout": None,
    "warmup_ratio": None,
    "quantization_bit": None,
    "preprocessing_num_workers": None,
    "dataloader_num_workers": None,
    "bf16": None,
    "fp16": None,
}

YAML_CONFIG, OUTPUT_DIR, YAML_DATA = prepare_runtime_yaml(
    project_dir=PROJECT_DIR,
    base_yaml_config=BASE_YAML_CONFIG,
    runtime_yaml_config=RUNTIME_YAML_CONFIG,
    use_override=USE_RUNTIME_YAML_OVERRIDE,
    overrides=YAML_OVERRIDES,
    strict_keys=True,
)

# Cập nhật biến môi trường
mlflow_tags = json.loads(os.environ["MLFLOW_TAGS"])
mlflow_tags.update({
    "dataset": str(YAML_DATA.get("dataset", "")),
    "yaml_config": YAML_CONFIG,
    "yaml_override": str(USE_RUNTIME_YAML_OVERRIDE).lower(),
})

os.environ.update({
    "YAML_CONFIG": YAML_CONFIG,
    "OUTPUT_DIR": OUTPUT_DIR,
    "MLFLOW_TAGS": json.dumps(mlflow_tags),
})

Runtime YAML Override: ON
YAML gốc: /kaggle/working/test-unimer/train/Uni-MuMER-train.yaml
YAML dùng để train: /kaggle/working/runtime_Uni-MuMER-train.yaml
Các key đã đổi:
  max_samples: 2 -> 3


In [7]:
%%bash
# 6. Kiểm tra DagsHub trước khi train
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"

🏃 View run bald-snake-98 at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1/runs/6d886a9541b04107b67688a6a7ed4784
🧪 View experiment at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1
DagsHub MLflow connection OK: https://dagshub.com/NhatPot/test-unimer.mlflow


In [8]:
%%bash
# 7. Training
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

echo "Python: $(command -v python)"
echo "Torchrun: $(command -v torchrun)"
MPLBACKEND=Agg llamafactory-cli train "$YAML_CONFIG" "run_name=uni-mumer-${RUN_UUID:0:8}"

Python: /kaggle/working/miniconda/envs/unimumer/bin/python
Torchrun: /kaggle/working/miniconda/envs/unimumer/bin/torchrun
[INFO|2026-06-24 13:20:47] llamafactory.launcher:143 >> Initializing 2 distributed tasks at: 127.0.0.1:54281
[WARNING|2026-06-24 13:20:57] llamafactory.hparams.parser:148 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-06-24 13:20:57] llamafactory.hparams.parser:143 >> Set `ddp_find_unused_parameters` to False in DDP training since LoRA is enabled.
[INFO|2026-06-24 13:20:57] llamafactory.hparams.parser:465 >> Process rank: 0, world size: 2, device: cuda:0, distributed training: True, compute dtype: torch.float16
[INFO|2026-06-24 13:20:57] llamafactory.hparams.parser:465 >> Process rank: 1, world size: 2, device: cuda:1, distributed training: True, compute dtype: torch.float16
[INFO|2026-06-24 13:21:01] llamafactory.data.loader:143 >> Loading dataset phxember/Uni-MuMER-Data...
training example:
input_ids:
[151644, 8948, 198, 2610, 525, 264

W0624 13:20:49.122000 407 site-packages/torch/distributed/run.py:792] 
W0624 13:20:49.122000 407 site-packages/torch/distributed/run.py:792] *****************************************
W0624 13:20:49.122000 407 site-packages/torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0624 13:20:49.122000 407 site-packages/torch/distributed/run.py:792] *****************************************
[INFO|tokenization_utils_base.py:2023] 2026-06-24 13:20:58,596 >> loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3/vocab.json
[INFO|tokenization_utils_base.py:2023] 2026-06-24 13:20:58,596 >> loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b82

In [ ]:
%%bash
# 8. Full Benchmark Test (3 CROHME datasets)
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

# Tìm checkpoint cuối cùng
LAST_CKPT=$(ls -td "$OUTPUT_DIR"/checkpoint-* 2>/dev/null | head -1)

if [[ -z "$LAST_CKPT" ]]; then
  echo "ERROR: No checkpoint found in $OUTPUT_DIR"
  exit 1
fi

echo "Testing with checkpoint: $LAST_CKPT"
echo "Running full benchmark on 3 CROHME datasets (3,332 samples)..."
echo ""

# Chạy full benchmark
python scripts/kaggle_full_test.py \
  --base-model Qwen/Qwen2.5-VL-3B-Instruct \
  --adapter-path "$LAST_CKPT" \
  --test-datasets crohme_2014 crohme_2016 crohme_2019 \
  --backup-dir example_data/backup \
  --base-results-dir example_data/CROHME/results \
  --output-dir kaggle_test_results \
  --project-dir "$PROJECT_DIR" \
  --batch-size 2

# In summary
echo ""
echo "============================================================"
echo "                    TEST SUMMARY"
echo "============================================================"
for dataset in crohme_2014 crohme_2016 crohme_2019; do
  echo ""
  echo "=== $dataset ==="
  if [[ -f "kaggle_test_results/${dataset}_results.txt" ]]; then
    cat "kaggle_test_results/${dataset}_results.txt" | grep -E "(Mean Edit Score|BLEU-4|Character Error Rate|Exact Match)" | head -4
  else
    echo "Results not found"
  fi
done

echo ""
echo "============================================================"
echo "Full comparison table:"
cat kaggle_test_results/comparison_table.txt

In [ ]:
%%bash
# 9. Upload artifacts và test results lên DagsHub
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

# Upload model checkpoint và config
python scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --notebook "$NOTEBOOK_PATH"

# Upload test results nếu có
if [[ -d "kaggle_test_results" ]]; then
  echo "Uploading test results to DagsHub..."

  python -c "import os, mlflow; mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI']); mlflow.start_run(run_id=os.environ['RUN_UUID']); mlflow.log_artifacts('kaggle_test_results', artifact_path='test_results'); mlflow.end_run(); print('✓ Test results uploaded')"
fi

In [22]:
%%bash
cd /kaggle/working/test-unimer

echo "Đang ở:"
pwd

echo "Branch hiện tại:"
git branch --show-current

echo "Trạng thái trước khi pull:"
git status --short

echo "Pull code mới nhất:"
git pull

Đang ở:
/kaggle/working/test-unimer
Branch hiện tại:
main
Trạng thái trước khi pull:
 M saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora/README.md
 M saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora/adapter_config.json
 M saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora/all_results.json
 M saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora/train_results.json
 M saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora/trainer_log.jsonl
 M saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora/trainer_state.json
 M saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora/training_loss.png
?? kaggle_test_results/
Pull code mới nhất:
Updating 0010743..e5d0d9f
Fast-forward
 scripts/kaggle_test_adapter.py | 12 ++++++------
 1 file changed, 6 insertions(+), 6 deletions(-)


From https://github.com/NhatPot/test-unimer
   0010743..e5d0d9f  main       -> origin/main


In [24]:
%%bash
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

# Kiểm tra transformers version
python -c "import transformers; print('Transformers version:', transformers.__version__)"

# Kiểm tra các class Qwen có sẵn
python -c "from transformers import AutoConfig; config = AutoConfig.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct', trust_remote_code=True); print('Model type:', config.model_type); print('Architectures:', config.architectures)"

# Kiểm tra xem có Qwen2_5_VLForConditionalGeneration không
python -c "try:
    from transformers import Qwen2_5_VLForConditionalGeneration
    print('✓ Qwen2_5_VLForConditionalGeneration available')
except ImportError as e:
    print('✗ Qwen2_5_VLForConditionalGeneration NOT available:', e)
"

# Kiểm tra xem có AutoModelForVision2Seq không
python -c "try:
    from transformers import AutoModelForVision2Seq
    print('✓ AutoModelForVision2Seq available')
except ImportError as e:
    print('✗ AutoModelForVision2Seq NOT available:', e)
"


Transformers version: 4.52.4
Model type: qwen2_5_vl
Architectures: ['Qwen2_5_VLForConditionalGeneration']
✓ Qwen2_5_VLForConditionalGeneration available
✓ AutoModelForVision2Seq available


In [25]:
%%bash
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python << 'EOF'
from transformers import AutoConfig
import json

config = AutoConfig.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct', trust_remote_code=True)

print("="*60)
print("Model Config:")
print("="*60)
print(f"model_type: {config.model_type}")
print(f"architectures: {config.architectures}")
print(f"auto_map: {config.auto_map if hasattr(config, 'auto_map') else 'N/A'}")
print("="*60)
EOF


Model Config:
model_type: qwen2_5_vl
architectures: ['Qwen2_5_VLForConditionalGeneration']
auto_map: N/A


In [28]:
%%bash
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python << 'EOF'
import torch

print("\n" + "="*60)
print("Testing different model classes:")
print("="*60)

# Test 1: Qwen2VLForConditionalGeneration (hiện tại đang dùng - SAI)
print("\n1. Testing Qwen2VLForConditionalGeneration:")
try:
    from transformers import Qwen2VLForConditionalGeneration
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2.5-VL-3B-Instruct',
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )
    print(f"   ✓ Loaded successfully")
    print(f"   Model class: {type(model).__name__}")
    print(f"   Model dtype: {model.dtype}")
    del model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ✗ Failed: {e}")

# Test 2: Qwen2_5_VLForConditionalGeneration (ĐÚNG nếu có)
print("\n2. Testing Qwen2_5_VLForConditionalGeneration:")
try:
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2.5-VL-3B-Instruct',
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )
    print(f"   ✓ Loaded successfully")
    print(f"   Model class: {type(model).__name__}")
    print(f"   Model dtype: {model.dtype}")
    del model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ✗ Failed: {e}")

# Test 3: AutoModelForVision2Seq
print("\n3. Testing AutoModelForVision2Seq:")
try:
    from transformers import AutoModelForVision2Seq
    model = AutoModelForVision2Seq.from_pretrained(
        'Qwen/Qwen2.5-VL-3B-Instruct',
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )
    print(f"   ✓ Loaded successfully")
    print(f"   Model class: {type(model).__name__}")
    print(f"   Model dtype: {model.dtype}")
    del model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ✗ Failed: {e}")

# Test 4: AutoModel (fallback)
print("\n4. Testing AutoModel:")
try:
    from transformers import AutoModel
    model = AutoModel.from_pretrained(
        'Qwen/Qwen2.5-VL-3B-Instruct',
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )
    print(f"   ✓ Loaded successfully")
    print(f"   Model class: {type(model).__name__}")
    print(f"   Model dtype: {model.dtype}")
    del model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ✗ Failed: {e}")

print("\n" + "="*60)
EOF



Testing different model classes:

1. Testing Qwen2VLForConditionalGeneration:
   ✗ Failed: Error(s) in loading state_dict for Linear:
	size mismatch for bias: copying a param with shape torch.Size([2048]) from checkpoint, the shape in current model is torch.Size([1280]).

2. Testing Qwen2_5_VLForConditionalGeneration:
   ✓ Loaded successfully
   Model class: Qwen2_5_VLForConditionalGeneration
   Model dtype: torch.float16

3. Testing AutoModelForVision2Seq:
   ✓ Loaded successfully
   Model class: Qwen2_5_VLForConditionalGeneration
   Model dtype: torch.float16

4. Testing AutoModel:
   ✓ Loaded successfully
   Model class: Qwen2_5_VLModel
   Model dtype: torch.float16



You are using a model of type qwen2_5_vl to instantiate a model of type qwen2_vl. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.33s/it]


In [29]:
%%bash
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python << 'EOF'
import torch
from transformers import BitsAndBytesConfig

print("\n" + "="*60)
print("Testing 4-bit quantization:")
print("="*60)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Quantization config:")
print(f"  load_in_4bit: {quantization_config.load_in_4bit}")
print(f"  bnb_4bit_quant_type: {quantization_config.bnb_4bit_quant_type}")
print(f"  bnb_4bit_compute_dtype: {quantization_config.bnb_4bit_compute_dtype}")
print(f"  bnb_4bit_use_double_quant: {quantization_config.bnb_4bit_use_double_quant}")

# Check CUDA
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Check bitsandbytes
try:
    import bitsandbytes as bnb
    print(f"\nbitsandbytes version: {bnb.__version__}")
except:
    print("\n✗ bitsandbytes not available")

print("="*60)
EOF



Testing 4-bit quantization:
Quantization config:
  load_in_4bit: True
  bnb_4bit_quant_type: nf4
  bnb_4bit_compute_dtype: torch.bfloat16
  bnb_4bit_use_double_quant: True

CUDA available: True
CUDA device: Tesla T4
CUDA version: 12.4

bitsandbytes version: 0.49.2
